[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/14_bootstrapping.ipynb)

# Bootstrapping — rejection sampling, distillation

> **Fine-tuning series — 14 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 10. Bootstrapping: rejection sampling · distillation

Teach a model from *generated* data. **Rejection-sampling fine-tuning** (RFT / RAFT /
STaR): sample many candidates, keep the best by a reward/verifier, SFT on the winners.
**Knowledge distillation**: train a small student to imitate a bigger teacher — TRL's
`GKDTrainer` does the on-policy / sequence-level version.

**Rejection sampling (best-of-N → SFT)** — a recipe, no special trainer.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)

def verifier(text):                       # toy: prefer short answers; swap for real check
    return -len(text)

def best_of_n(prompt, n=4):
    # return_dict=True so we can splat into generate(); transformers 5.x no longer
    # returns a bare tensor from apply_chat_template(return_tensors="pt").
    enc = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
            add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
    outs = model.generate(**enc, max_new_tokens=32, do_sample=True, temperature=1.0,
                          num_return_sequences=n)
    plen = enc["input_ids"].shape[1]
    cands = [tokenizer.decode(o[plen:], skip_special_tokens=True) for o in outs]
    return max(cands, key=verifier)

winners = [{"instruction": e["instruction"], "output": best_of_n(e["instruction"])}
           for e in instructions[:5]]
rft_ds = Dataset.from_list([to_chat_text(w) for w in winners])
print(winners[0])
# Then fine-tune on rft_ds with ANY how-axis method above (e.g. SFTTrainer + lora_cfg).
del model; cleanup()

**Knowledge distillation (GKD)** — student imitates teacher; use a *bigger* teacher in practice.

In [ ]:
try:
    from trl import GKDConfig, GKDTrainer
except ImportError:
    from trl.experimental.gkd import GKDConfig, GKDTrainer

student = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32)
TEACHER_ID = MODEL_ID                      # in practice: a larger instructed model

gkd_ds = Dataset.from_list([{"messages": [
    {"role": "user", "content": e["instruction"]},
    {"role": "assistant", "content": e["output"]}]} for e in instructions])

gkd_trainer = GKDTrainer(
    model=student, teacher_model=TEACHER_ID,
    args=GKDConfig(output_dir="ft_gkd", per_device_train_batch_size=2, max_steps=20,
        learning_rate=2e-5, lmbda=0.5, beta=0.5, seq_kd=False,
        max_new_tokens=32, bf16=BF16_OK, logging_steps=5, report_to="none"),
    train_dataset=gkd_ds, processing_class=tokenizer)
gkd_trainer.train(); del gkd_trainer, student; cleanup()
# Hard/offline alternative: let the teacher GENERATE answers, then plain-SFT the
# student on them — i.e. rejection sampling with teacher as the generator.